In [1]:
import pandas as pd
import geopandas as gpd
import json
import requests
from shapely.geometry.linestring import LineString, Point

In [2]:
df = pd.read_parquet("../data/trips_pointv3_cleaned.parquet")
# try with a single trip
df = df[df.unique_id=='24c83c4e-1e60-4b65-9100-8cf1550076df_2022-01-07']
# sort pts by order in time
df = df.sort_values(by=['point_timestamp'])
# geodataframe
df = gpd.GeoDataFrame(
    df,
    crs='EPSG:4326',
    geometry=gpd.points_from_xy(df.point_longitude, df.point_latitude))

In [3]:
gdf_linestring = LineString(df.geometry.to_list()).wkt # linestring of the trip

In [4]:
# all points in the trip
gdf_points_temp = df.apply(lambda x: [y for y in x['geometry'].coords], axis=1)
gdf_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy([a_tuple[0] for a_tuple in gdf_points_temp[0]], [a_tuple[1] for a_tuple in gdf_points_temp[0]]), crs=4326)

# geodataframe of linestring and pts
gdf = gpd.GeoDataFrame(pd.concat([df, gdf_points], ignore_index=True))

In [5]:
# dataframe with pts lat and long (for the single trip chosen)
df_points = pd.DataFrame({'lon':df.point_longitude, 'lat':df.point_latitude})
df_points.head(3)


,lon,lat
1449,11.118389,46.054911
1450,11.118389,46.054911
1451,11.119268,46.054611


In [6]:
# request
meili_coordinates = df_points.to_json(orient='records')
meili_head = '{"shape":'
meili_tail = ""","search_radius": 300, "shape_match":"map_snap", "costing":"multimodal", "format":"osrm"}"""
# https://github.com/valhalla/valhalla-docs/blob/master/turn-by-turn/api-reference.md
# search radius -> distance to look between points and road network; costing -> auto, bicycle, ...
meili_request_body = meili_head + meili_coordinates + meili_tail
url = "http://localhost:8002/trace_route"
# curl http://localhost:8002/route --data '{"locations":[{"lat":48.632608,"lon":9.010350,"type":"break"},{"lat":48.631776,"lon":9.011528,"type":"break"}],"costing":"auto","directions_options":{"units":"miles"}}' | jq '.'
headers = {'Content-type': 'application/json'}
data = str(meili_request_body)
r = requests.post(url, data=data, headers=headers)

In [9]:
# get response
if r.status_code == 200:
    response_text = json.loads(r.text)
else:
    print('Error',r.status_code,':\n',r.text)
    raise ValueError
search_1 = response_text.get('matchings')
search_2 = dict(search_1[0])
polyline6 = search_2.get('geometry')
search_3 = response_text.get('tracepoints')

Error 400 :
 {"code":"NoSegment","message":"One of the supplied input coordinates could not snap to street segment."}


ValueError: 